In [11]:
import sys
from podio import root_io
import math
path = "/work/eic/users/aabhishe/EIC_3He_5x41_XRot_pi_bc.hepmc3.tree_sim_output_recon.root"

In [20]:


# 1. Open your EIC output file
reader = root_io.Reader(path)

# Counters
total_events = 0
events_with_truth_match = 0
events_with_proton = 0
total_protons_found = 0

# 2. Loop over events
for ievt, event in enumerate(reader.get("events")):
    total_events += 1
    
    hits = event.get("ForwardRomanPotHits")
    if not hits:
        continue

    event_has_truth_match = False
    
    # Track unique protons seen in this specific event to avoid
    # counting multiple detector layers for the same particle
    found_proton_ids_in_event = set()

    for hit in hits:
        # Filter out low-energy secondary background (delta-rays, soft photons)
        if hit.getMomentum().z < 20.0:
            continue

        mc_particle = hit.getParticle()

        # Check if truth particle information is linked
        if mc_particle.isAvailable():
            event_has_truth_match = True
            
            # Check for proton (PDG = 2212)
            if mc_particle.getPDG() == 2212:
                # Use the Podio/EDM object ID or address to distinguish unique particles
                proton_id = mc_particle.getObjectID()
                found_proton_ids_in_event.add(proton_id)

    if event_has_truth_match:
        events_with_truth_match += 1

    if found_proton_ids_in_event:
        events_with_proton += 1
        total_protons_found += len(found_proton_ids_in_event)

# 3. Print Summary Results
print("=" * 45)
print("             ANALYSIS SUMMARY")
print("=" * 45)
print(f"Total events analyzed:                 {total_events}")
print(f"Events with truth-matched hits:        {events_with_truth_match}")
print(f"Events with at least one proton:       {events_with_proton}")
print(f"Total unique spectator protons found:  {total_protons_found}")
if total_events > 0:
    print(f"Proton Roman Pot Acceptance/Hit Rate:  {100.0 * events_with_proton / total_events:.2f}%")
print("=" * 45)

             ANALYSIS SUMMARY
Total events analyzed:                 1000
Events with truth-matched hits:        772
Events with at least one proton:       363
Total unique spectator protons found:  475
Proton Roman Pot Acceptance/Hit Rate:  36.30%
